# 第2章：搭建 Kaggle 开发环境

> **预计学习时间：约 30 分钟**
>
> 本章将带你理解为什么训练大模型需要 GPU，如何在免费的 Kaggle 平台上搭建开发环境，以及显存管理的核心技巧。

**运行环境**: Kaggle Notebook (免费 T4 GPU)
**前置要求**: 注册 Kaggle 账号，创建新 Notebook

---
## 1. GPU vs CPU：为什么深度学习需要 GPU？

### 类比：一个专家 vs 一支军队

想象你要把 10000 个包裹贴上标签：

```
CPU 方式（一个专家工人）:
  包裹 1 → 包裹 2 → 包裹 3 → ...
  每个 1 秒，10,000 个包裹 → 10,000 秒

GPU 方式（5,000 个工人同时工作）:
  工人 1：包裹 1  |
  工人 2：包裹 2  | 全部同时进行！
  工人 3：包裹 3  |
  ...x 5000       |
  10,000 个包裹 → 2 秒！
```

![CPU vs GPU](images/cpu_vs_gpu.png)

*图1：CPU vs GPU 架构对比*

**核心差异**：CPU 有 4-16 个强大核心（擅长复杂逻辑），GPU 有数千个简单核心（擅长并行计算）。

### 为什么深度学习偏爱 GPU？

深度学习的本质是**大量矩阵运算**，而矩阵运算中的乘法之间**互不依赖**，完美适合 GPU 并行。

| 模型 | 矩阵大小 | 乘法次数 |
|------|----------|----------|
| GPT-2 | 768 x 768 | ~60 万 |
| LLaMA-7B | 4096 x 4096 | ~1700 万 |
| LLaMA-70B | 8192 x 8192 | ~6700 万 |
| DeepSeek | 7168 x 18432 | ~1.3 亿 |

每生成一个词都要算一次！生成 100 个词 = 上面的计算 x 100。

![批处理](images/batch_processing.png)

*图2：批处理 vs 串行处理*

### 真实性能对比

| 硬件 | 训练耗时 | 加速比 |
|------|----------|--------|
| CPU (i7-12700) | 48 小时 | 1x |
| GPU (T4, Kaggle) | 30 分钟 | ~100x |
| GPU (A100) | 5 分钟 | ~600x |

**动手验证**：先检查一下我们的系统环境。

In [ ]:
import platform
import sys
import os

print("=" * 50)
print("系统环境信息")
print("=" * 50)

# --- 操作系统信息 ---
print(f"操作系统: {platform.system()} {platform.release()}")
print(f"Python 版本: {sys.version}")
print(f"Python 路径: {sys.executable}")
print()

# --- CPU 信息 ---
# Kaggle 通常提供 4 核 CPU
cpu_count = os.cpu_count()
print(f"CPU 核心数: {cpu_count}")
print()

# --- 内存信息 ---
# 读取 /proc/meminfo 获取系统内存（Linux 环境）
try:
    with open('/proc/meminfo', 'r') as f:
        for line in f:
            if 'MemTotal' in line:
                # 将 KB 转换为 GB
                mem_kb = int(line.split()[1])
                print(f"系统内存: {mem_kb / 1024 / 1024:.1f} GB")
                break
except FileNotFoundError:
    print("无法读取内存信息（非 Linux 环境）")

# --- 磁盘空间 ---
import shutil
total, used, free = shutil.disk_usage("/")
print(f"磁盘总空间: {total / 1024**3:.1f} GB")
print(f"磁盘可用空间: {free / 1024**3:.1f} GB")

### 检查 GPU 硬件

用 `nvidia-smi` 命令查看 GPU 信息：

In [ ]:
# --- 使用 nvidia-smi 查看 GPU 硬件信息 ---
# nvidia-smi 是 NVIDIA 提供的 GPU 管理命令行工具
# 它能显示 GPU 型号、显存、温度、功耗等信息
!nvidia-smi

**如何阅读 nvidia-smi 输出**：GPU Name（型号）、Memory（显存）、GPU-Util（利用率）、Temp（温度）。

接下来用 PyTorch 检查 CUDA 是否可用：

In [ ]:
import torch

print("=" * 50)
print("PyTorch GPU 检查")
print("=" * 50)

# --- 检查 CUDA 是否可用 ---
# CUDA 是 NVIDIA 的并行计算平台，PyTorch 通过它与 GPU 通信
print(f"PyTorch 版本: {torch.__version__}")
print(f"CUDA 是否可用: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"CUDA 版本: {torch.version.cuda}")
    print(f"cuDNN 版本: {torch.backends.cudnn.version()}")
    print()
    
    # --- GPU 详细信息 ---
    gpu_count = torch.cuda.device_count()
    print(f"可用 GPU 数量: {gpu_count}")
    print()
    
    for i in range(gpu_count):
        props = torch.cuda.get_device_properties(i)
        print(f"--- GPU {i} ---")
        print(f"  名称: {props.name}")
        print(f"  显存总量: {props.total_memory / 1024**3:.1f} GB")
        print(f"  多处理器数量: {props.multi_processor_count}")
        print(f"  计算能力: {props.major}.{props.minor}")
    
    # 设置默认设备
    device = torch.device("cuda")
    print(f"\n已设置默认设备: {device}")
else:
    print("\n[警告] CUDA 不可用！")
    print("请检查：")
    print("  1. Kaggle Notebook 右侧面板 → Accelerator → 选择 GPU T4 x2")
    print("  2. 重启 Notebook")
    device = torch.device("cpu")
    print(f"\n回退到 CPU: {device}")

> 💡 **记忆要点**
> - CPU = 少量强大核心（4-16 个），擅长复杂逻辑；GPU = 大量简单核心（数千个），擅长并行计算
> - 深度学习的核心是矩阵运算，矩阵运算可以高度并行 → GPU 天生适合
> - GPU 训练速度比 CPU 快 **50-100 倍**以上

---
## 2. 显存管理：GPU 上最珍贵的资源

### 什么是显存（VRAM）？

显存是 GPU 自己专用的内存，和电脑的内存（RAM）是分开的。

```
Your Computer:

  +-------+          +---------+
  |  CPU  |          |   GPU   |
  |       |          |         |
  |  RAM  |          |  VRAM   |
  | 16-64 |          |  4-80   |
  |   GB  |          |   GB    |
  +-------+          +---------+

它们是分开的！GPU 只能使用 VRAM 中的数据。
要在 GPU 上计算，数据必须从 RAM 复制到 VRAM。
```

![显存层级](images/memory_hierarchy.png)

*图3：GPU 显存层级结构*

### 显存里放了什么？

训练一个模型时，显存中需要存放四样东西（以 7B 参数模型为例）：

| 内容 | 大小 | 类比 |
|------|------|------|
| 模型参数 | 7B x 2 字节 (FP16) = 14 GB | 教科书 |
| 梯度 | 和参数一样大 = 14 GB | 老师的批改 |
| 优化器状态 | Adam 需要 2 倍 = 28 GB | 错题本 |
| 中间激活值 | 取决于 batch size | 草稿纸 |
| **总计** | **~60-80 GB** | Kaggle T4 只有 16 GB！ |

### 省显存的常见技巧

![显存管理](images/memory_management.png)

*图4：GPU 显存管理的四大策略*

1. **减小 Batch Size** — 一次放不下 32 个样本？那就一次放 8 个
2. **混合精度训练** — FP32 (4 bytes) → FP16 (2 bytes)，省一半！类比："3.141592653589793" 简写成 "3.14"
3. **梯度累积** — 想要 batch=32 但只够 batch=8？分 4 轮计算再汇总
4. **梯度检查点** — 不保存所有中间结果，需要时重新计算。用时间换空间

### OOM 错误

```
RuntimeError: CUDA out of memory.（显存不足！）
尝试分配 2.00 GiB
(GPU 0; 总容量 15.90 GiB; 已分配 12.34 GiB)
```

**口诀：OOM 了？batch 减半！还 OOM？精度减半！还 OOM？模型减半！**

**动手验证**：让我们观察显存的使用情况。

In [ ]:
import torch

if torch.cuda.is_available():
    print("=" * 50)
    print("GPU 显存信息")
    print("=" * 50)
    
    # --- 显存总量 ---
    total_mem = torch.cuda.get_device_properties(0).total_memory
    print(f"显存总量: {total_mem / 1024**3:.2f} GB")
    
    # --- 当前显存使用情况 ---
    allocated = torch.cuda.memory_allocated(0)
    reserved = torch.cuda.memory_reserved(0)
    print(f"已分配显存: {allocated / 1024**2:.1f} MB")
    print(f"已预留显存: {reserved / 1024**2:.1f} MB")
    print(f"可用显存: {(total_mem - allocated) / 1024**3:.2f} GB")
    
    print("\n--- 显存占用估算 ---")
    print(f"{'模型大小':>12} {'FP32 参数':>12} {'训练总计':>12} {'T4 能否训练':>12}")
    print("-" * 52)
    for name, params in [("10M", 10e6), ("100M", 100e6), ("300M", 300e6),
                          ("1B", 1e9), ("3B", 3e9)]:
        # FP32 参数占用: params * 4 bytes
        param_gb = params * 4 / 1024**3
        # 训练总计 ≈ 参数 + 梯度 + 优化器 ≈ 4倍参数
        train_gb = param_gb * 4
        can_train = "可以" if train_gb < 14 else "不行"
        print(f"{name:>12} {param_gb:>10.1f} GB {train_gb:>10.1f} GB {can_train:>12}")
    
    print("\n提示: 使用 FP16 半精度可以将显存占用减半!")
else:
    print("CUDA 不可用，无法查看 GPU 显存信息")

In [ ]:
# --- 实验: 分配大量显存，观察显存变化 ---
if torch.cuda.is_available():
    print("=" * 50)
    print("显存动态监控实验")
    print("=" * 50)
    
    # 清空缓存
    torch.cuda.empty_cache()
    print(f"清空缓存后 - 已分配: {torch.cuda.memory_allocated(0) / 1024**2:.1f} MB")
    
    # 创建一个大张量（约 1GB）
    # 256M 个 float32 数字 = 256M * 4 bytes = 1 GB
    print("\n创建 1GB 大张量...")
    big_tensor = torch.randn(256_000_000, device="cuda")
    print(f"分配后 - 已分配: {torch.cuda.memory_allocated(0) / 1024**2:.1f} MB")
    
    # 删除张量
    del big_tensor
    print(f"删除后 - 已分配: {torch.cuda.memory_allocated(0) / 1024**2:.1f} MB")
    
    # 清空 CUDA 缓存
    # 注意: del 只是释放引用，torch 可能仍保留缓存
    # empty_cache() 才真正释放给系统
    torch.cuda.empty_cache()
    print(f"清空缓存后 - 已预留: {torch.cuda.memory_reserved(0) / 1024**2:.1f} MB")
    
    print("\n关键要点:")
    print("  - del 释放 Python 引用，但 PyTorch 可能保留缓存")
    print("  - torch.cuda.empty_cache() 释放未使用的缓存")
    print("  - 训练时如果遇到 OOM (Out of Memory)，")
    print("    可以减小 batch_size 或使用 FP16")

> 💡 **记忆要点**
> - 显存（VRAM）是 GPU 专用内存，训练时最紧缺的资源
> - 训练时要放 4 样东西：**模型参数、梯度、优化器状态、中间激活值**
> - OOM 是最常见的错误，解决方法：**减 batch size、用混合精度、梯度累积**

---
## 3. Kaggle 平台：免费的 GPU 训练场

### 为什么选择 Kaggle？

| 方案 | 费用 | GPU | 适合人群 |
|------|------|-----|----------|
| 自购显卡 | $700+ 一次性 | RTX 3090, 16-24GB | 长期使用者 |
| 云服务 (AWS/Azure) | $1-7/小时 | A100, 40-80GB | 企业用户 |
| Google Colab | 免费/付费 | T4/A100, 15GB | 学习者 |
| **Kaggle** | **免费** | **T4 x 2, 15GB x 2** | **学习者（推荐！）** |

### Kaggle 提供什么？

![Kaggle 工作流](images/kaggle_workflow.png)

*图5：Kaggle 开发工作流*

- **GPU 配额**：每周 30 小时，可选 T4 x 1 或 T4 x 2，每个 Session 最长 12 小时
- **T4 GPU**：16 GB 显存，2560 CUDA 核心，支持混合精度加速
- **其他**：13 GB RAM，~70 GB 磁盘，预装 Python/PyTorch/TensorFlow

### Kaggle 使用技巧

- 写代码和调试时**关闭 GPU**（用 CPU 就够了），只在训练时开 GPU
- 训练中定期保存 checkpoint，Session 超时后可以恢复
- 16 GB 显存可以微调 0.5B-3B 参数模型（用 LoRA），本课程完全够用

### 软件栈

```
你的 Python 代码
    |
    v
PyTorch（深度学习框架）
    |   model.to("cuda")  ← 将模型移到 GPU
    v
CUDA（NVIDIA 的 GPU 编程平台）
    |   转换为 GPU 并行指令
    v
GPU 硬件（数千个核心同时计算）
```

### 安装必要的库

In [ ]:
# --- 检查已安装的核心库 ---
# 这些库在后续章节中会反复使用

libraries = {
    "torch": "深度学习框架，所有模型构建和训练的基础",
    "transformers": "Hugging Face 的预训练模型库，加载/使用各种大模型",
    "datasets": "Hugging Face 的数据集库，方便加载和处理训练数据",
    "numpy": "数值计算库，处理数组和矩阵运算",
    "pandas": "数据分析库，处理表格数据",
    "matplotlib": "可视化库，绘制图表",
    "tokenizers": "高性能分词器库，用于构建自定义 Tokenizer",
    "accelerate": "Hugging Face 的训练加速库，简化多 GPU/混合精度训练",
    "tqdm": "进度条库，训练时显示进度",
}

print("=" * 60)
print("核心库版本检查")
print("=" * 60)
print(f"{'库名':<18} {'版本':>12}  {'说明'}")
print("-" * 60)

missing = []
for lib, desc in libraries.items():
    try:
        mod = __import__(lib)
        version = getattr(mod, '__version__', '未知')
        print(f"{lib:<18} {version:>12}  {desc}")
    except ImportError:
        print(f"{lib:<18} {'[未安装]':>12}  {desc}")
        missing.append(lib)

if missing:
    print(f"\n需要安装: {', '.join(missing)}")
else:
    print(f"\n所有核心库均已安装!")

In [ ]:
# --- 安装缺失的库（如果有的话）---
# Kaggle 通常已预装大部分库，但以防万一，运行以下命令
# -q 参数表示静默模式，减少输出信息

!pip install -q transformers datasets tokenizers accelerate
print("安装完成!")

> 💡 **记忆要点**
> - Kaggle 提供**免费** T4 GPU，每周 30 小时，对学习完全够用
> - 最重要的限制是 **16 GB 显存**和**每周 30 小时**
> - CUDA 是连接 PyTorch 和 GPU 硬件的桥梁

---
## 4. 验证 PyTorch + CUDA

光看版本号还不够，让我们实际运行 GPU 计算来验证环境真正可用。

In [ ]:
import torch
import time

print("=" * 50)
print("PyTorch + CUDA 功能验证")
print("=" * 50)

# --- 测试1: 在 GPU 上创建张量 ---
print("\n[测试1] 在 GPU 上创建张量")
if torch.cuda.is_available():
    # 在 GPU 上创建一个随机张量
    x = torch.randn(3, 3, device="cuda")
    print(f"  张量形状: {x.shape}")
    print(f"  所在设备: {x.device}")
    print(f"  数据类型: {x.dtype}")
    print(f"  张量内容:\n{x}")
    print("  [通过]")
else:
    print("  [跳过] CUDA 不可用")

# --- 测试2: CPU 与 GPU 之间的数据传输 ---
print("\n[测试2] CPU <-> GPU 数据传输")
if torch.cuda.is_available():
    # 在 CPU 上创建张量
    cpu_tensor = torch.randn(1000, 1000)
    print(f"  CPU 张量: {cpu_tensor.device}")
    
    # 移到 GPU
    gpu_tensor = cpu_tensor.to("cuda")
    print(f"  GPU 张量: {gpu_tensor.device}")
    
    # 移回 CPU
    back_to_cpu = gpu_tensor.to("cpu")
    print(f"  回到 CPU: {back_to_cpu.device}")
    
    # 验证数据一致性
    assert torch.allclose(cpu_tensor, back_to_cpu)
    print("  数据一致性: 通过")
    print("  [通过]")
else:
    print("  [跳过] CUDA 不可用")

# --- 测试3: GPU 上的基本运算 ---
print("\n[测试3] GPU 矩阵运算")
if torch.cuda.is_available():
    a = torch.randn(256, 256, device="cuda")
    b = torch.randn(256, 256, device="cuda")
    
    # 矩阵乘法
    c = torch.matmul(a, b)
    print(f"  矩阵乘法: ({a.shape}) x ({b.shape}) = ({c.shape})")
    print(f"  结果设备: {c.device}")
    print("  [通过]")
else:
    print("  [跳过] CUDA 不可用")

# --- 测试4: 梯度计算（反向传播的基础）---
print("\n[测试4] 自动求导（Autograd）")
if torch.cuda.is_available():
    # 创建需要梯度的张量
    x = torch.tensor([2.0, 3.0], device="cuda", requires_grad=True)
    # 计算 y = x^2 的和
    y = (x ** 2).sum()
    # 反向传播，计算梯度
    y.backward()
    # dy/dx = 2x，所以梯度应该是 [4.0, 6.0]
    print(f"  x = {x.data}")
    print(f"  y = x^2 之和 = {y.item()}")
    print(f"  dy/dx = {x.grad}  (应该是 2*x = [4.0, 6.0])")
    assert torch.allclose(x.grad, 2 * x.data)
    print("  [通过]")
else:
    print("  [跳过] CUDA 不可用")

print("\n" + "=" * 50)
print("所有测试通过! PyTorch + CUDA 环境正常")
print("=" * 50)

---
## 5. 性能实测：CPU vs GPU

矩阵乘法是深度学习最核心的运算——全连接层、注意力机制、卷积层，本质上都是矩阵乘法。让我们实测对比：

In [ ]:
import torch
import time

def benchmark_matmul(size, device, n_runs=10):
    """测试指定大小矩阵乘法的平均耗时"""
    # 创建两个随机矩阵
    a = torch.randn(size, size, device=device)
    b = torch.randn(size, size, device=device)
    
    # GPU 需要先预热（第一次运行通常较慢）
    if device == "cuda":
        torch.matmul(a, b)
        torch.cuda.synchronize()  # 等待 GPU 完成
    
    # 正式计时
    times = []
    for _ in range(n_runs):
        if device == "cuda":
            torch.cuda.synchronize()
        start = time.perf_counter()
        
        c = torch.matmul(a, b)
        
        if device == "cuda":
            torch.cuda.synchronize()  # GPU 运算是异步的，需要同步才能准确计时
        end = time.perf_counter()
        times.append(end - start)
    
    avg_time = sum(times) / len(times)
    return avg_time


# --- 不同矩阵大小的对比 ---
sizes = [512, 1024, 2048, 4096]

print("=" * 65)
print("CPU vs GPU 矩阵乘法性能对比")
print("=" * 65)
print(f"{'矩阵大小':>12} {'CPU 耗时':>12} {'GPU 耗时':>12} {'加速比':>10}")
print("-" * 48)

for size in sizes:
    cpu_time = benchmark_matmul(size, "cpu", n_runs=5)
    
    if torch.cuda.is_available():
        gpu_time = benchmark_matmul(size, "cuda", n_runs=10)
        speedup = cpu_time / gpu_time
        print(f"{size}x{size:>5} {cpu_time*1000:>10.2f}ms {gpu_time*1000:>10.2f}ms {speedup:>9.1f}x")
    else:
        print(f"{size}x{size:>5} {cpu_time*1000:>10.2f}ms {'N/A':>12} {'N/A':>10}")

print("\n结论: 矩阵越大，GPU 的加速效果越明显!")
print("这就是为什么深度学习离不开 GPU。")

**观察**：小矩阵时 GPU 优势不明显（数据传输开销），大矩阵时 GPU 快 10-100 倍。

### FP32 vs FP16 性能对比

FP16（半精度）在 T4 GPU 上有专门的硬件加速单元（Tensor Cores）：

In [ ]:
# --- FP32 vs FP16 性能对比 ---
# FP16（半精度）在 T4 GPU 上有专门的硬件加速单元（Tensor Cores）

if torch.cuda.is_available():
    print("=" * 55)
    print("FP32 vs FP16 性能对比（GPU 上）")
    print("=" * 55)
    
    size = 4096
    n_runs = 20
    
    # FP32（单精度，默认）
    a32 = torch.randn(size, size, device="cuda", dtype=torch.float32)
    b32 = torch.randn(size, size, device="cuda", dtype=torch.float32)
    torch.matmul(a32, b32); torch.cuda.synchronize()  # 预热
    
    times_32 = []
    for _ in range(n_runs):
        torch.cuda.synchronize()
        t0 = time.perf_counter()
        torch.matmul(a32, b32)
        torch.cuda.synchronize()
        times_32.append(time.perf_counter() - t0)
    avg_32 = sum(times_32) / len(times_32)
    
    # FP16（半精度）
    a16 = a32.half()  # .half() 即 FP32 -> FP16
    b16 = b32.half()
    torch.matmul(a16, b16); torch.cuda.synchronize()  # 预热
    
    times_16 = []
    for _ in range(n_runs):
        torch.cuda.synchronize()
        t0 = time.perf_counter()
        torch.matmul(a16, b16)
        torch.cuda.synchronize()
        times_16.append(time.perf_counter() - t0)
    avg_16 = sum(times_16) / len(times_16)
    
    print(f"矩阵大小: {size}x{size}")
    print(f"FP32 耗时: {avg_32 * 1000:.2f} ms")
    print(f"FP16 耗时: {avg_16 * 1000:.2f} ms")
    print(f"FP16 加速比: {avg_32 / avg_16:.1f}x")
    print(f"\nFP32 显存: {a32.element_size() * a32.nelement() * 2 / 1024**2:.0f} MB (两个矩阵)")
    print(f"FP16 显存: {a16.element_size() * a16.nelement() * 2 / 1024**2:.0f} MB (两个矩阵)")
    print(f"\n结论: FP16 不仅更快，还省一半显存!")
    print(f"这就是混合精度训练的核心优势。")
    
    # 清理
    del a32, b32, a16, b16
    torch.cuda.empty_cache()

---
## 6. GPU 训练流程测试

用一个最简单的神经网络（学习 y = 2x + 1）来验证完整训练流程：

1. **定义模型** → 2. **准备数据** → 3. **前向传播** → 4. **计算损失** → 5. **反向传播** → 6. **更新参数**

In [ ]:
import torch
import torch.nn as nn
import time

# --- 定义一个简单的神经网络 ---
# 这个网络有 3 层，学习一个简单的函数: y = 2x + 1
class SimpleNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(1, 64),    # 输入层: 1 -> 64
            nn.ReLU(),            # 激活函数
            nn.Linear(64, 64),   # 隐藏层: 64 -> 64
            nn.ReLU(),
            nn.Linear(64, 1),    # 输出层: 64 -> 1
        )
    
    def forward(self, x):
        return self.layers(x)


# --- 选择设备 ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"训练设备: {device}")

# --- 创建模型和优化器 ---
model = SimpleNet().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
criterion = nn.MSELoss()  # 均方误差损失

param_count = sum(p.numel() for p in model.parameters())
print(f"模型参数量: {param_count}")

# --- 生成训练数据 ---
# 目标函数: y = 2x + 1 + 噪声
n_samples = 1000
X = torch.randn(n_samples, 1, device=device)
Y = 2 * X + 1 + 0.1 * torch.randn(n_samples, 1, device=device)

# --- 训练循环 ---
print("\n开始训练...")
start_time = time.time()
losses = []

for epoch in range(200):
    # 前向传播: 模型做预测
    predictions = model(X)
    
    # 计算损失: 预测值与真实值的差距
    loss = criterion(predictions, Y)
    
    # 反向传播: 计算每个参数的梯度
    optimizer.zero_grad()  # 清零上一步的梯度
    loss.backward()        # 反向传播
    
    # 更新参数: 根据梯度调整参数
    optimizer.step()
    
    losses.append(loss.item())
    
    if (epoch + 1) % 50 == 0:
        print(f"  Epoch {epoch+1:>4}/200  Loss: {loss.item():.6f}")

elapsed = time.time() - start_time
print(f"\n训练完成! 耗时: {elapsed:.2f} 秒")
print(f"最终损失: {losses[-1]:.6f}")

# --- 验证模型学到了什么 ---
model.eval()
with torch.no_grad():
    test_x = torch.tensor([[0.0], [1.0], [2.0], [5.0]], device=device)
    test_y = model(test_x)
    
print("\n--- 模型预测 vs 真实值 ---")
print(f"{'输入 x':>8} {'模型预测':>10} {'真实值 (2x+1)':>14}")
print("-" * 35)
for x_val, y_pred in zip(test_x, test_y):
    true_y = 2 * x_val.item() + 1
    print(f"{x_val.item():>8.1f} {y_pred.item():>10.4f} {true_y:>14.4f}")

In [ ]:
# --- 可视化训练过程 ---
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 左图: 损失曲线
axes[0].plot(losses, color="steelblue", linewidth=1.5)
axes[0].set_title("训练损失曲线", fontsize=14)
axes[0].set_xlabel("训练轮次")
axes[0].set_ylabel("MSE 损失")
axes[0].set_yscale("log")  # 用对数坐标看得更清楚
axes[0].grid(True, alpha=0.3)

# 右图: 模型拟合效果
model.eval()
with torch.no_grad():
    x_plot = torch.linspace(-3, 3, 100, device=device).unsqueeze(1)
    y_plot = model(x_plot).cpu().numpy()

# 画散点（训练数据的子集）
axes[1].scatter(
    X[:200].cpu().numpy(), Y[:200].cpu().numpy(),
    alpha=0.3, s=10, color="gray", label="训练数据"
)
# 画模型预测线
axes[1].plot(
    x_plot.cpu().numpy(), y_plot,
    color="red", linewidth=2, label="模型预测"
)
# 画真实函数
axes[1].plot(
    x_plot.cpu().numpy(), 2 * x_plot.cpu().numpy() + 1,
    color="blue", linewidth=2, linestyle="--", label="真实值: y=2x+1"
)
axes[1].set_title("模型拟合 vs 真实函数", fontsize=14)
axes[1].set_xlabel("x")
axes[1].set_ylabel("y")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("模型成功学会了 y = 2x + 1 这个函数!")

---
## 7. 环境信息汇总

In [ ]:
import torch
import platform
import sys

print("=" * 55)
print("环境信息汇总 (保存此信息以备排查问题)")
print("=" * 55)

info = {
    "操作系统": f"{platform.system()} {platform.release()}",
    "Python": sys.version.split()[0],
    "PyTorch": torch.__version__,
    "CUDA 可用": str(torch.cuda.is_available()),
}

if torch.cuda.is_available():
    info.update({
        "CUDA 版本": torch.version.cuda,
        "cuDNN 版本": str(torch.backends.cudnn.version()),
        "GPU 型号": torch.cuda.get_device_name(0),
        "GPU 显存": f"{torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB",
        "GPU 数量": str(torch.cuda.device_count()),
    })

# 检查额外的库
for lib in ["transformers", "datasets", "tokenizers", "accelerate"]:
    try:
        mod = __import__(lib)
        info[lib] = mod.__version__
    except ImportError:
        info[lib] = "未安装"

for key, val in info.items():
    print(f"  {key:<18} {val}")

print("\n" + "=" * 55)
print("环境搭建完成! 你已经准备好开始学习了!")
print("=" * 55)

---
## 8. 本章总结

**1. GPU vs CPU** — CPU = 少量强核心（复杂逻辑），GPU = 大量弱核心（并行计算）。深度学习 = 大量矩阵运算 = GPU 的主场

**2. 显存管理** — 显存 = GPU 专用内存。训练占用：参数 + 梯度 + 优化器 + 激活值。省显存：减 batch、混合精度、梯度累积

**3. Kaggle 平台** — 免费 T4 GPU（16GB 显存），每周 30 小时，完全够用

**4. 软件栈** — Python → PyTorch → CUDA → GPU。`torch.cuda.is_available()` 是第一步验证

### 下一章预告

在第3章中，我们将学习**张量与神经网络基础**：
- 什么是张量
- 神经网络的本质：函数拟合
- 前向传播和反向传播
- 用 PyTorch 实现简单的线性回归

---

> **参考资料**
> - NVIDIA CUDA Programming Guide
> - PyTorch CUDA Semantics
> - Kaggle Documentation
> - *Mixed Precision Training* (Micikevicius et al., 2018)